# Simulated dataset: points per frame and optical flow

Sanity check of `realistic_dataset_v1` before it is used to train PWC-Net.

1. **Cohort**: clips, frames and frame pairs per patient. Frame counts differ between clips and patients.
2. **Points per frame**: the stored myocardial scatterers on top of the B-mode frames.
3. **Optical flow**: scatterer displacement between two frames, as sparse vectors and as a dense motion field (DMF). The DMF comes from the scatterers in the same way as the paper's training targets.
4. **Does it work?** Two checks of the dense field:
   - warping frame *t+1* back onto frame *t*;
   - chaining the flows over the cycle and comparing the result with the true scatterer positions.

Reads the cache `artifacts/simulated-cache/`, built by `python -m motion_estimator.scripts.build_simulated_cache`. It holds frames resampled to 0.26 mm, uint8 with 0 outside the sector, and 20k scatterers stored as 1-based pixels.

In [ ]:
import re, sys, zipfile
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "motion_estimator").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import hsv_to_rgb
from scipy.interpolate import LinearNDInterpolator
from scipy.ndimage import map_coordinates
from scipy.spatial import Delaunay, cKDTree
from IPython.display import HTML

# set directly: importing the pwcnet package pulls in torch, whose OpenMP runtime clashes with scipy's
SIM_CACHE = ROOT / "motion_estimator" / "artifacts" / "simulated-cache"

plt.rcParams["animation.embed_limit"] = 60  # MB
print(SIM_CACHE)

## 1. Cohort: frames per clip and per patient

Frame counts are read from the `.npz` headers, so no pixels are decompressed. A clip of *n* frames gives *n − 1* consecutive training pairs.

In [ ]:
NAME = re.compile(r"(?P<sim_type>.+_simulations)_(?P<view>\dch)_sim(?P<sim>\d+)")

def n_frames(path):
    with zipfile.ZipFile(path) as z, z.open("frames.npy") as f:
        major, _ = np.lib.format.read_magic(f)
        read = np.lib.format.read_array_header_1_0 if major == 1 else np.lib.format.read_array_header_2_0
        shape, _, _ = read(f)
    return shape[0]

rows = []
for path in sorted(SIM_CACHE.glob("Pat_*/*.npz")):
    m = NAME.fullmatch(path.stem)
    rows.append({"patient": path.parent.name, **m.groupdict(), "frames": n_frames(path), "path": path})
clips = pd.DataFrame(rows)
clips["pairs"] = clips.frames - 1

per_patient = (clips.groupby("patient")
               .agg(clips=("frames", "size"), frames=("frames", "sum"), pairs=("pairs", "sum"),
                    min_frames=("frames", "min"), max_frames=("frames", "max"))
               .sort_values("frames", ascending=False))
print(f"{len(clips)} clips, {clips.patient.nunique()} patients, "
      f"{clips.frames.sum()} frames, {clips.pairs.sum()} consecutive pairs")
print(clips.groupby(["sim_type", "view"]).agg(clips=("frames", "size"), frames=("frames", "sum")))
per_patient

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [1, 3]})
axes[0].hist(clips.frames, bins=range(clips.frames.min(), clips.frames.max() + 2), color="#4c72b0")
axes[0].set_xlabel("frames per clip"); axes[0].set_ylabel("clips")
axes[1].bar(per_patient.index, per_patient.frames, color="#4c72b0")
axes[1].set_ylabel("frames per patient"); axes[1].tick_params(axis="x", rotation=90, labelsize=7)
for ax in axes:
    ax.grid(alpha=0.25, axis="y")
fig.tight_layout()

## 2. One clip: points per frame

Set `CLIP` to a `"Pat_<id>/<file stem>"` key, or leave it as `None` to take the first clip. Scatterers are converted to **0-based** `(x = column, y = row)`. ED and ES are stored **1-based** in the cache and converted here.

In [ ]:
CLIP = None  # e.g. "Pat_108/regional_simulations_2ch_sim1"

row = clips.iloc[0] if CLIP is None else clips[(clips.patient + "/" + clips.path.map(lambda p: p.stem)) == CLIP].iloc[0]
with np.load(row.path, allow_pickle=False) as d:
    frames = d["frames"].astype(np.float32)
    sx = d["scat_x"].astype(np.float64) - 1.0   # 1-based -> 0-based
    sy = d["scat_y"].astype(np.float64) - 1.0
    ed, es = int(d["ed_frame"]) - 1, int(d["es_frame"]) - 1
    info = {k: d[k].item() for k in ("frame_interval_s", "ejection_fraction", "heart_rate")}
    pixel_mm = float(d["pixel_mm"][0])

T, H, W = frames.shape
sector = (frames > 0).all(axis=0)   # 0 is reserved for "outside the sector"
MARGIN = 30
x_lim = (max(sx.min() - MARGIN, 0), min(sx.max() + MARGIN, W - 1))
y_lim = (min(sy.max() + MARGIN, H - 1), max(sy.min() - MARGIN, 0))   # reversed: image rows go down
print(f"{row.patient}/{row.path.stem}: {T} frames, {H}x{W} px @ {pixel_mm} mm, "
      f"{sx.shape[1]} scatterers, ED={ed}, ES={es} (0-based)")
print({k: round(v, 3) for k, v in info.items()})

def show_frame(ax, t, title=None):
    ax.imshow(frames[t], cmap="gray", vmin=1, vmax=255)
    ax.set_xlim(x_lim); ax.set_ylim(y_lim); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title if title is not None else f"frame {t}", fontsize=9)

In [ ]:
# Evenly subsampled scatterers, coloured by their depth at ED, so a point keeps its colour as it moves
SHOW = np.linspace(0, sx.shape[1] - 1, 600).astype(int)
colour = sy[ed, SHOW]

picks = sorted({0, ed, (ed + es) // 2, es, min(es + (T - es) // 2, T - 1), T - 1})
fig, axes = plt.subplots(1, len(picks), figsize=(3.2 * len(picks), 3.4))
for ax, t in zip(axes, picks):
    tag = " (ED)" if t == ed else " (ES)" if t == es else ""
    show_frame(ax, t, f"frame {t}{tag}")
    ax.scatter(sx[t, SHOW], sy[t, SHOW], c=colour, cmap="turbo", s=3, linewidths=0)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5), dpi=80)
show_frame(ax, 0)
image = ax.images[0]
points = ax.scatter(sx[0, SHOW], sy[0, SHOW], c=colour, cmap="turbo", s=3, linewidths=0)

def update(t):
    image.set_data(frames[t])
    points.set_offsets(np.c_[sx[t, SHOW], sy[t, SHOW]])
    ax.set_title(f"frame {t}" + (" (ED)" if t == ed else " (ES)" if t == es else ""), fontsize=9)
    return image, points

anim = animation.FuncAnimation(fig, update, frames=T, interval=80)
plt.close(fig)
HTML(anim.to_jshtml())

## 3. Scatterer motion over the cycle

On the left are the trajectories of 80 scatterers drawn on the ED frame. On the right is the displacement per frame step over **all** scatterers. Motion is about 1 px per step, so consecutive pairs are small-displacement problems.

In [ ]:
step = np.hypot(np.diff(sx, axis=0), np.diff(sy, axis=0))   # (T-1, n_scatterers), px per frame step
rr, cc = np.round(sy).astype(int).clip(0, H - 1), np.round(sx).astype(int).clip(0, W - 1)
in_sector = sector[rr, cc].sum(axis=1)
print(f"scatterers per frame: {sx.shape[1]} stored in every frame; "
      f"inside the sector {in_sector.min()}-{in_sector.max()}")

TRAJ = np.linspace(0, sx.shape[1] - 1, 80).astype(int)
norm = plt.Normalize(colour.min(), colour.max())
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.7]})
show_frame(axes[0], ed, f"trajectories over {T} frames, on ED")
for i in TRAJ:
    axes[0].plot(sx[:, i], sy[:, i], lw=0.9, color=plt.cm.turbo(norm(sy[ed, i])))
axes[0].scatter(sx[ed, TRAJ], sy[ed, TRAJ], s=6, c="white", zorder=3)

t_mid = np.arange(T - 1) + 0.5
ax = axes[1]
ax.plot(t_mid, np.median(step, axis=1), lw=2, label="median")
ax.plot(t_mid, np.percentile(step, 95, axis=1), lw=2, label="p95")
ax.axvline(ed, color="gray", ls="--"); ax.text(ed, ax.get_ylim()[1], " ED", va="top", fontsize=8)
ax.axvline(es, color="gray", ls="--"); ax.text(es, ax.get_ylim()[1], " ES", va="top", fontsize=8)
ax.set_xlabel("frame"); ax.set_ylabel("displacement per step (px)")
ax.secondary_yaxis("right", functions=(lambda p: p * pixel_mm, lambda m: m / pixel_mm)).set_ylabel("mm")
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()

## 4. Optical flow from the scatterers

For a pair *(t0, t1)*, each scatterer's displacement is interpolated linearly (Delaunay) onto the pixel grid of frame *t0*. The result is the **forward** flow: pixel *x* in *t0* moves to *x + f(x)* in *t1*.

- **Myocardium mask**: pixels within 2 px of a scatterer. The median spacing between scatterers is ~0.7 px. Mask pixels just outside the scatterer hull take the displacement of their nearest scatterer.
- **Outside the mask** the linear interpolation fills the cavity across the hull. That part is not ground truth and is never displayed or scored.

Flows are computed on the crop around the LV and cached, so every pair is computed once. Colour coding: hue gives the direction (wheel inset) and brightness the magnitude.

In [ ]:
xa, xb = int(np.floor(x_lim[0])), int(np.ceil(x_lim[1]))
ya, yb = int(np.floor(y_lim[1])), int(np.ceil(y_lim[0]))
CROP = (slice(ya, yb + 1), slice(xa, xb + 1))
EXTENT = (xa - 0.5, xb + 0.5, yb + 0.5, ya - 0.5)
GY, GX = np.mgrid[ya:yb + 1, xa:xb + 1]
GRID = np.c_[GX.ravel(), GY.ravel()]
MASK_RADIUS = 2.0

_tris, _flows = {}, {}

def dense_flow(t0, t1):
    # (h, w, 2) flow on the crop and its (h, w) myocardium mask
    if (t0, t1) not in _flows:
        pts = np.c_[sx[t0], sy[t0]]
        if t0 not in _tris:
            _tris[t0] = Delaunay(pts)
        disp = np.c_[sx[t1] - sx[t0], sy[t1] - sy[t0]]
        flow = LinearNDInterpolator(_tris[t0], disp)(GRID)
        dist, nearest = cKDTree(pts).query(GRID, distance_upper_bound=MASK_RADIUS + 1e-6)
        mask = dist <= MASK_RADIUS
        fill = mask & np.isnan(flow[:, 0])
        flow[fill] = disp[nearest[fill]]
        _flows[(t0, t1)] = (flow.reshape(GY.shape + (2,)).astype(np.float32), mask.reshape(GY.shape))
    return _flows[(t0, t1)]

def flow_to_rgb(flow, max_mag=None):
    u, v = flow[..., 0], flow[..., 1]
    mag = np.hypot(u, v)
    max_mag = max_mag or np.nanmax(mag)
    hsv = np.stack([(np.arctan2(v, u) + np.pi) / (2 * np.pi), np.ones_like(u), np.clip(mag / max_mag, 0, 1)], axis=-1)
    return hsv_to_rgb(np.nan_to_num(hsv))

def show_flow(ax, flow, mask, max_mag=None):
    ax.imshow(np.dstack([flow_to_rgb(flow, max_mag), mask.astype(float)]), extent=EXTENT)

def colour_wheel(ax):
    yy, xx = np.mgrid[-1:1:101j, -1:1:101j]   # rows go down, like the image
    rgb = flow_to_rgb(np.dstack([xx, yy]), 1.0)
    ax.imshow(np.dstack([rgb, (np.hypot(xx, yy) <= 1).astype(float)]))
    ax.set_axis_off()

In [ ]:
PAIR_T = None   # first frame of the pair to show; None -> the step with the largest median motion
t0 = int(np.argmax(np.median(step, axis=1))) if PAIR_T is None else PAIR_T
flow, mask = dense_flow(t0, t0 + 1)
mag = np.where(mask, np.hypot(flow[..., 0], flow[..., 1]), np.nan)
GAIN = 6
QUIV = np.linspace(0, sx.shape[1] - 1, 350).astype(int)

fig, axes = plt.subplots(1, 4, figsize=(19, 4.8))
show_frame(axes[0], t0, f"frame {t0} -> {t0 + 1}")
show_frame(axes[1], t0, f"scatterer displacement (x{GAIN})")
axes[1].quiver(sx[t0, QUIV], sy[t0, QUIV], sx[t0 + 1, QUIV] - sx[t0, QUIV], sy[t0 + 1, QUIV] - sy[t0, QUIV],
               color="yellow", angles="xy", scale_units="xy", scale=1 / GAIN, width=0.003)
show_frame(axes[2], t0, "dense flow (myocardium mask)")
show_flow(axes[2], flow, mask)
colour_wheel(axes[2].inset_axes([0.78, 0.02, 0.2, 0.2]))
show_frame(axes[3], t0, "flow magnitude")
im = axes[3].imshow(mag, extent=EXTENT, cmap="magma")
fig.colorbar(im, ax=axes[3], fraction=0.046, label="px / frame")
fig.tight_layout()
print(f"pair {t0}->{t0 + 1}: {mask.sum()} mask px, magnitude median {np.nanmedian(mag):.2f} px, "
      f"max {np.nanmax(mag):.2f} px ({np.nanmax(mag) * pixel_mm:.2f} mm)")

### Dense flow over the cycle

Consecutive pairs spread over the clip share one colour scale (p99 magnitude across them), so brightness can be compared between pairs.

In [ ]:
CYCLE = np.unique(np.linspace(0, T - 2, 10).astype(int))
fields = {t: dense_flow(t, t + 1) for t in CYCLE}
shared = np.nanpercentile(np.concatenate([np.hypot(f[m, 0], f[m, 1]) for f, m in fields.values()]), 99)

fig, axes = plt.subplots(2, (len(CYCLE) + 1) // 2, figsize=(3.2 * ((len(CYCLE) + 1) // 2), 7))
for ax, t in zip(axes.ravel(), CYCLE):
    tag = " (ED)" if t == ed else " (ES)" if t == es else ""
    show_frame(ax, t, f"{t} -> {t + 1}{tag}")
    show_flow(ax, *fields[t], max_mag=shared)
for ax in axes.ravel()[len(CYCLE):]:
    ax.set_axis_off()
colour_wheel(axes.ravel()[0].inset_axes([0.75, 0.02, 0.23, 0.23]))
fig.suptitle(f"full brightness = {shared:.2f} px / frame", fontsize=10)
fig.tight_layout()

## 5. Check 1: does the flow explain the images?

If the flow is right, frame *t+1* sampled at *x + f(x)* should look like frame *t* at *x* (backward warp). Mean absolute intensity difference is computed inside the myocardium mask, with and without the warp. The warped error should stay clearly below the zero-flow error, and the gap should grow where motion is large. It will not reach zero: speckle decorrelates, and the frames are resampled and quantised to uint8.

In [ ]:
def warp_back(t0, t1):
    flow, mask = dense_flow(t0, t1)
    f = np.nan_to_num(flow)
    warped = map_coordinates(frames[t1], [GY + f[..., 1], GX + f[..., 0]], order=1, mode="nearest")
    return warped, mask

records = []
for t in range(T - 1):
    warped, m = warp_back(t, t + 1)
    i0, i1 = frames[t][CROP], frames[t + 1][CROP]
    records.append({"t": t, "no_flow": np.abs(i0 - i1)[m].mean(), "warped": np.abs(i0 - warped)[m].mean(),
                    "median_motion_px": np.median(step[t])})
photo = pd.DataFrame(records).set_index("t")
photo["reduction_%"] = 100 * (1 - photo.warped / photo.no_flow)

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(photo.index + 0.5, photo.no_flow, lw=2, label="|I0 - I1| (no flow)")
ax.plot(photo.index + 0.5, photo.warped, lw=2, label="|I0 - warp(I1)| (dense flow)")
for t, name in ((ed, "ED"), (es, "ES")):
    ax.axvline(t, color="gray", ls="--"); ax.text(t, ax.get_ylim()[1], f" {name}", va="top", fontsize=8)
ax.set_xlabel("frame"); ax.set_ylabel("mean abs difference (uint8 levels)")
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()
photo.describe().round(2)

In [ ]:
warped, m = warp_back(t0, t0 + 1)
i0, i1 = frames[t0][CROP], frames[t0 + 1][CROP]
before = np.where(m, np.abs(i0 - i1), np.nan)
after = np.where(m, np.abs(i0 - warped), np.nan)
vmax = np.nanpercentile(before, 99)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
show_frame(axes[0], t0, f"frame {t0}")
for ax, img, title in ((axes[1], before, "|I0 - I1|"), (axes[2], after, "|I0 - warp(I1)|")):
    show_frame(ax, t0, f"{title}, mean {np.nanmean(img):.1f}")
    im = ax.imshow(img, extent=EXTENT, cmap="inferno", vmin=0, vmax=vmax)
fig.colorbar(im, ax=axes[1:], fraction=0.03, label="uint8 levels")

## 6. Check 2: chaining the flows over the cycle

This is the paper's contour propagation applied to the ground-truth flow. 200 evenly spaced scatterers start at frame 0. At each step their positions are moved by the dense flow *t → t+1*, sampled bilinearly at the current position, and then compared with the **true** scatterer positions.

Drift comes only from rasterising the flow and accumulating the error, so it should stay well below a pixel over the clip. A larger drift means the interpolation or the coordinate convention is wrong. A point is *lost* when the flow under it is undefined, i.e. it left the scatterer hull.

In [ ]:
TRACK = np.linspace(0, sx.shape[1] - 1, 200).astype(int)
est = np.full((T, len(TRACK), 2), np.nan)
est[0] = np.c_[sx[0, TRACK], sy[0, TRACK]]
for t in range(T - 1):
    flow, _ = dense_flow(t, t + 1)
    coords = [est[t, :, 1] - ya, est[t, :, 0] - xa]
    u = map_coordinates(flow[..., 0], coords, order=1, mode="constant", cval=np.nan)
    v = map_coordinates(flow[..., 1], coords, order=1, mode="constant", cval=np.nan)
    est[t + 1] = est[t] + np.c_[u, v]

truth = np.stack([sx[:, TRACK], sy[:, TRACK]], axis=-1)
drift = np.hypot(*(est - truth).transpose(2, 0, 1))   # (T, n_points), NaN once lost
lost = np.isnan(drift).sum(axis=1)

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(np.nanmedian(drift, axis=1), lw=2, label="median")
ax.plot(np.nanpercentile(drift, 90, axis=1), lw=2, label="p90")
ax.plot(np.nanmax(drift, axis=1), lw=1, color="gray", label="max")
for t, name in ((ed, "ED"), (es, "ES")):
    ax.axvline(t, color="gray", ls="--"); ax.text(t, ax.get_ylim()[1], f" {name}", va="top", fontsize=8)
ax.set_xlabel("frame"); ax.set_ylabel("drift from true position (px)")
ax.secondary_yaxis("right", functions=(lambda p: p * pixel_mm, lambda m: m / pixel_mm)).set_ylabel("mm")
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()
print(f"at ES: median {np.nanmedian(drift[es]):.3f} px, p90 {np.nanpercentile(drift[es], 90):.3f} px; "
      f"last frame: median {np.nanmedian(drift[-1]):.3f} px; lost points: {lost[-1]}/{len(TRACK)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, t in zip(axes, (es, T - 1)):
    show_frame(ax, t, f"frame {t}" + (" (ES)" if t == es else " (last)"))
    ax.scatter(truth[t, :, 0], truth[t, :, 1], s=22, facecolors="none", edgecolors="white", linewidths=0.8,
               label="true scatterer")
    ax.scatter(est[t, :, 0], est[t, :, 1], s=5, c="red", label="chained dense flow")
axes[0].legend(loc="lower right", fontsize=8, frameon=False, labelcolor="white")
fig.tight_layout()

## What to look for

- **Points per frame**: the dots sit on the myocardium in every frame and move with it, with no jumps, flips or offset. If they are offset, check the 1-based → 0-based conversion. If they are rotated, check the orientation.
- **Motion curve**: there should be a peak in systole, a pause, then filling. A flat or noisy curve means the scatterer tracks are wrong.
- **Dense flow**: the field is smooth along the wall and the colours turn with the wall: base and apex move differently, and the direction reverses between systole and diastole.
- **Check 1**: the warped error should stay below the no-flow error for every pair.
- **Check 2**: drift should be sub-pixel with no lost points. Then the ground-truth dense field is safe to use as a PWC-Net training target, and the chaining code can score PWC-Net's predicted flows the same way.